# 07 — ML Classification (Paris)

Trains Logistic Regression, XGBoost, and Random Forest to classify
grid cells as **residential**, **commercial**, or **industrial**.

**Features (X):**
- `amenity_density`, `amenity_ratio_food_drink` (OSM)
- `avg_height`, `avg_floors`, `avg_construction_year`, `building_count`, `residential_ratio` (EUBUCCO)
- `landuse_entropy` (EUBUCCO)
- `tourism_density` (OSM)
- `shop_density_km2`, `brand_ratio` (OSM)

**Target (Y):** `zone_type` from notebook 01 (residential / commercial / industrial)

**Input:** `csv/Paris/combined_grid.csv`

**Output:** plots to `outputs/Paris/`, predictions to `csv/Paris/07_predictions.csv`

In [1]:
PARIS_CONFIG = "paris.json"
PLOTS_DIR    = "outputs/Paris"

In [2]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json
import os

sns.set(color_codes=True, rc={"figure.figsize": (10, 8)})
os.makedirs(PLOTS_DIR, exist_ok=True)

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CSV_DIR = config["csv_dir"]
CSV_PATH = f"{CSV_DIR}/combined_grid.csv"

df = pd.read_csv(CSV_PATH, dtype={"cell_id": str})
print(f"Loaded {len(df)} rows x {df.shape[1]} cols")
print(f"\nZone type distribution:")
print(df["zone_type"].value_counts().to_string())

Loaded 120331 rows x 16 cols

Zone type distribution:
zone_type
residential    101989
industrial      10458
commercial       7458
mixed             426


In [3]:
# ── Keep only target labels ───────────────────────────
TARGET_LABELS = config["target_labels"]  # ["residential", "commercial", "industrial"]
df = df[df["zone_type"].isin(TARGET_LABELS)].copy()
df["label"] = df["zone_type"]
print(f"After filtering to target labels: {len(df)} cells")
print(df["label"].value_counts().to_string())

After filtering to target labels: 119905 cells
label
residential    101989
industrial      10458
commercial       7458


In [4]:
# ── EDA: class distribution ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="label", ax=ax)
ax.set_title("Paris — Zone Type Distribution")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/01_countplot_zone_type.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\2239869048.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# ── Feature columns ───────────────────────────────────
FEATURE_COLS = [
    "amenity_density",
    "amenity_ratio_food_drink",
    "avg_height",
    "avg_floors",
    "avg_construction_year",
    "building_count",
    "residential_ratio",
    "landuse_entropy",
    "tourism_density",
    "shop_density_km2",
    "brand_ratio",
]

# Only use columns that exist
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f"Features used: {FEATURE_COLS}")

Features used: ['amenity_density', 'amenity_ratio_food_drink', 'avg_height', 'avg_floors', 'avg_construction_year', 'building_count', 'residential_ratio', 'landuse_entropy', 'tourism_density', 'shop_density_km2', 'brand_ratio']


In [6]:
# ── EDA: feature boxplots ─────────────────────────────
n_feats = len(FEATURE_COLS)
n_cols  = 3
n_rows  = (n_feats + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
fig.suptitle("Paris — Feature Distributions by Zone Type", fontsize=14)
for ax, feat in zip(axes.flatten(), FEATURE_COLS):
    sns.boxplot(data=df, x="label", y=feat, ax=ax)
    ax.set_title(feat)
for ax in axes.flatten()[n_feats:]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/02_feature_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\195895857.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── Correlation heatmap ───────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
corr = df[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/03_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\3621076297.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ── Prepare X and Y ───────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

X = df[FEATURE_COLS].copy().fillna(df[FEATURE_COLS].median())
encoder = LabelEncoder()
y = encoder.fit_transform(df["label"])
class_names = list(encoder.classes_)

print(f"Classes:  {class_names}")
print(f"X shape:  {X.shape}")

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}   Test: {X_test.shape[0]}")

Classes:  ['commercial', 'industrial', 'residential']
X shape:  (119905, 11)
Train: 95924   Test: 23981


In [9]:
# ── Logistic Regression ───────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from mlxtend.plotting import plot_confusion_matrix

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
scores = cross_val_score(lr, X_train, y_train, cv=5)
print(f"LR Cross-val: {scores.mean():.3f} (+/- {scores.std():.3f})")

lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print(f"Test accuracy: {lr.score(X_test, y_test):.3f}")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

fig, ax = plot_confusion_matrix(conf_mat=confusion_matrix(y_test, y_pred_lr),
                                colorbar=True, show_absolute=True, show_normed=True,
                                class_names=class_names)
plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/04_confusion_matrix_lr.png", dpi=150, bbox_inches="tight")
plt.show()

LR Cross-val: 0.921 (+/- 0.003)


Test accuracy: 0.923
              precision    recall  f1-score   support

  commercial       0.45      0.61      0.52      1492
  industrial       0.71      0.78      0.74      2091
 residential       1.00      0.96      0.98     20398

    accuracy                           0.92     23981
   macro avg       0.72      0.78      0.75     23981
weighted avg       0.94      0.92      0.93     23981



C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\3504838235.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ── XGBoost ───────────────────────────────────────────
import xgboost as xgb

xgb_model = xgb.XGBClassifier(random_state=42, eval_metric="mlogloss")
scores = cross_val_score(xgb_model, X_train, y_train, cv=5)
print(f"XGB Cross-val: {scores.mean():.3f} (+/- {scores.std():.3f})")

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print(f"Test accuracy: {xgb_model.score(X_test, y_test):.3f}")
print(classification_report(y_test, y_pred_xgb, target_names=class_names))

fig, ax = plot_confusion_matrix(conf_mat=confusion_matrix(y_test, y_pred_xgb),
                                colorbar=True, show_absolute=True, show_normed=True,
                                class_names=class_names)
plt.title("XGBoost — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/05_confusion_matrix_xgb.png", dpi=150, bbox_inches="tight")
plt.show()

XGB Cross-val: 0.949 (+/- 0.001)


Test accuracy: 0.951
              precision    recall  f1-score   support

  commercial       0.69      0.57      0.63      1492
  industrial       0.76      0.81      0.78      2091
 residential       0.99      0.99      0.99     20398

    accuracy                           0.95     23981
   macro avg       0.81      0.79      0.80     23981
weighted avg       0.95      0.95      0.95     23981



C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\3167825069.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ── Random Forest ─────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight="balanced", random_state=42)
scores = cross_val_score(rf, X_train, y_train, cv=5)
print(f"RF Cross-val: {scores.mean():.3f} (+/- {scores.std():.3f})")

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f"Test accuracy: {rf.score(X_test, y_test):.3f}")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

fig, ax = plot_confusion_matrix(conf_mat=confusion_matrix(y_test, y_pred_rf),
                                colorbar=True, show_absolute=True, show_normed=True,
                                class_names=class_names)
plt.title("Random Forest — Confusion Matrix")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/06_confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.show()

RF Cross-val: 0.946 (+/- 0.001)


Test accuracy: 0.949
              precision    recall  f1-score   support

  commercial       0.67      0.59      0.62      1492
  industrial       0.76      0.77      0.77      2091
 residential       0.99      0.99      0.99     20398

    accuracy                           0.95     23981
   macro avg       0.80      0.78      0.79     23981
weighted avg       0.95      0.95      0.95     23981



C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\2690028101.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# ── Feature importance ────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind="barh", ax=ax)
ax.set_title("Random Forest — Feature Importance (Paris)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/07_feature_importance_rf.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\Hani\AppData\Local\Temp\ipykernel_3556\3502640230.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# ── Export predictions ────────────────────────────────
X_all        = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
X_all_scaled = scaler.transform(X_all)

df["predicted_zone"] = encoder.inverse_transform(rf.predict(X_all_scaled))
proba = rf.predict_proba(X_all_scaled)

for i, cls in enumerate(encoder.classes_):
    df[f"prob_{cls}"] = proba[:, i].round(4)
df["confidence"] = proba.max(axis=1).round(4)

pred_cols = ["cell_id", "cell_lat", "cell_lon", "zone_type", "predicted_zone", "confidence"] \
           + [f"prob_{c}" for c in encoder.classes_]
predictions = df[pred_cols]

pred_path = f"{CSV_DIR}/07_predictions.csv"
predictions.to_csv(pred_path, index=False, encoding="utf-8")
print(f"Saved predictions: {pred_path} ({len(predictions)} cells)")
print(predictions["predicted_zone"].value_counts().to_string())

Saved predictions: csv/Paris/07_predictions.csv (119905 cells)
predicted_zone
residential    102115
industrial      10556
commercial       7234
